# Stage 1 baseline / ablation (EfficientNet-B3 only)
نفس تقسيم Stage 1 المعتمد (من `stage1_split_regenerated.csv`) وبنفس الكود الأصلي. غيّري `VARIANT` في الخلية 1 وشغّلي كل مرة بمتغير واحد.

In [ ]:
# ========== 1) الإعدادات ==========
VARIANT = "plain"   # plain | no_sampler | no_labelsmooth | basic_aug   (full = الإعداد الأصلي)

VARIANTS = {
    "full":           dict(sampler=True,  ls=0.1, aug="full"),
    "plain":          dict(sampler=False, ls=0.0, aug="basic"),
    "no_sampler":     dict(sampler=False, ls=0.1, aug="full"),
    "no_labelsmooth": dict(sampler=True,  ls=0.0, aug="full"),
    "basic_aug":      dict(sampler=True,  ls=0.1, aug="basic"),
}
CFG = VARIANTS[VARIANT]

import os, glob, json, time, random, shutil, numpy as np, pandas as pd, torch, torch.nn as nn
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/skin_disease_project"
SPLIT_CSV = glob.glob("/content/drive/MyDrive/**/stage1_split_regenerated.csv", recursive=True)[0]
EXTERNAL_DIR = "/content/drive/MyDrive/AI-Leprosy.v1i.coco"
OUT_DIR = os.path.join(PROJECT_DIR, "ablation_stage1", VARIANT)
os.makedirs(OUT_DIR, exist_ok=True)
ORGANIZED = "/content/organized_data_ablation"

SEED = 42
IMAGE_SIZE, BATCH_SIZE, NUM_EPOCHS = 300, 16, 50
LEARNING_RATE, WEIGHT_DECAY, DROPOUT_RATE = 1e-4, 1e-4, 0.4
EARLY_STOPPING_PATIENCE, LR_SCHEDULER_PATIENCE, FREEZE_BACKBONE_EPOCHS = 8, 3, 5
LABEL_SMOOTHING = CFG["ls"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("VARIANT:", VARIANT, CFG, "| device:", DEVICE, "| split csv:", SPLIT_CSV)
assert DEVICE.type == "cuda", "فعّلي GPU: Runtime > Change runtime type > T4"

In [ ]:
# ========== 2) تجهيز الداتا من ملف التقسيم المعتمد (نفس تقسيم Stage 1 بالظبط) ==========
from concurrent.futures import ThreadPoolExecutor
sp = pd.read_csv(SPLIT_CSV)
print(sp.groupby(["split","label"]).size().unstack())
miss = [p for p in sp.path if not os.path.exists(p)]
assert not miss, f"ملفات مفقودة: {len(miss)} مثال: {miss[:2]}"
print("كل الصور موجودة:", len(sp))

if os.path.exists(ORGANIZED): shutil.rmtree(ORGANIZED)
jobs = []
for r in sp.itertuples():
    cls = os.path.basename(os.path.dirname(r.path)).replace(" ", "_")
    dest = os.path.join(ORGANIZED, r.split, r.label, f"{cls}__{os.path.basename(r.path)}")
    jobs.append((r.path, dest))
def _cp(j):
    os.makedirs(os.path.dirname(j[1]), exist_ok=True); shutil.copy2(j[0], j[1])
with ThreadPoolExecutor(max_workers=16) as ex: list(ex.map(_cp, jobs))
for s in ["train","val","test"]:
    print(s, {l: len(os.listdir(os.path.join(ORGANIZED,s,l))) for l in ["leprosy","non_leprosy"]})

In [ ]:
# ========== 3) Transforms + DataLoaders ==========
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def get_transforms():
    if CFG["aug"] == "full":
        train_tf = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5), transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(degrees=20),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
            transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.9,1.1)),
            transforms.ToTensor(), transforms.Normalize(MEAN, STD), transforms.RandomErasing(p=0.15)])
    else:
        train_tf = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    eval_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return train_tf, eval_tf

train_tf, eval_tf = get_transforms()
train_ds = datasets.ImageFolder(os.path.join(ORGANIZED,"train"), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(ORGANIZED,"val"),   transform=eval_tf)
test_ds  = datasets.ImageFolder(os.path.join(ORGANIZED,"test"),  transform=eval_tf)
CLASSES = train_ds.classes; LEP = CLASSES.index("leprosy"); print(CLASSES, "leprosy idx =", LEP)

if CFG["sampler"]:
    t = [l for _, l in train_ds.samples]; cc = torch.bincount(torch.tensor(t)).float()
    w = [(1.0/cc)[x] for x in t]
    sampler = WeightedRandomSampler(weights=w, num_samples=len(w), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ========== 4) الموديل + دوال التدريب (نفس الكود الأصلي لـ EfficientNet-B3) ==========
def build_model():
    m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    backbone = m.features; inf = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(p=DROPOUT_RATE, inplace=True), nn.Linear(inf, 2))
    return m, backbone

def set_trainable(b, flag):
    for p in b.parameters(): p.requires_grad = flag

class EarlyStopping:
    def __init__(self, patience, path): self.patience, self.path, self.best, self.c, self.stop = patience, path, np.inf, 0, False
    def __call__(self, v, model):
        if v < self.best:
            self.best, self.c = v, 0; torch.save(model.state_dict(), self.path); return True
        self.c += 1
        if self.c >= self.patience: self.stop = True
        return False

def run_epoch(model, loader, crit, opt, train):
    model.train() if train else model.eval(); tl=c=n=0; torch.set_grad_enabled(train)
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train: opt.zero_grad()
        o = model(x); loss = crit(o, y)
        if train: loss.backward(); opt.step()
        tl += loss.item()*x.size(0); c += (o.argmax(1)==y).sum().item(); n += y.size(0)
    return tl/n, 100*c/n
print("جاهز")

In [ ]:
# ========== 5) التدريب (حوالي ساعة - سيبيه شغال) ==========
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
model, backbone = build_model(); model.to(DEVICE)
crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
opt = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
sched = ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=LR_SCHEDULER_PATIENCE)
ckpt = os.path.join(OUT_DIR, "efficientnet_b3_best.pt")
es = EarlyStopping(EARLY_STOPPING_PATIENCE, ckpt)
set_trainable(backbone, False); t0 = time.time(); hist = []
for ep in range(1, NUM_EPOCHS+1):
    if ep == FREEZE_BACKBONE_EPOCHS+1: set_trainable(backbone, True)
    trl, tra = run_epoch(model, train_loader, crit, opt, True)
    vl, va = run_epoch(model, val_loader, crit, opt, False)
    sched.step(vl); better = es(vl, model); hist.append((ep, trl, tra, vl, va))
    print(f"Epoch {ep:03d} | train_loss={trl:.4f} acc={tra:.2f}% | val_loss={vl:.4f} acc={va:.2f}%{' *' if better else ''}")
    if es.stop: print("Early stopping at epoch", ep); break
print(f"انتهى في {(time.time()-t0)/60:.1f} دقيقة | best val_loss={es.best:.4f}")
pd.DataFrame(hist, columns=["epoch","train_loss","train_acc","val_loss","val_acc"]).to_csv(os.path.join(OUT_DIR,"history.csv"), index=False)

In [ ]:
# ========== 6) التقييم: test داخلي + الداتا الخارجية (120 صورة) + حفظ التنبؤات ==========
from PIL import Image
from sklearn.metrics import roc_auc_score
model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); model.to(DEVICE).eval()

@torch.no_grad()
def predict_paths(paths):
    out = []
    for i in range(0, len(paths), 16):
        x = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in paths[i:i+16]]).to(DEVICE)
        out.append(torch.softmax(model(x), 1)[:, LEP].cpu().numpy())
    return np.concatenate(out)

# داخلي
paths = [p for p, _ in test_ds.samples]; y = np.array([1 if l == LEP else 0 for _, l in test_ds.samples])
p_int = predict_paths(paths)
pd.DataFrame({"path": paths, "y": y, "p_leprosy": p_int}).to_csv(os.path.join(OUT_DIR, "test_probs.csv"), index=False)

# خارجي (نفس الـ 120 صورة المستخدمة في البحث)
EXT = json.loads('{"Leprosy_210823_195_jpg.rf.ac9a4362a0cba4c337de9e3790e8de4b.jpg": 1, "Leprosy_210823_102_jpg.rf.a90660d09edce9d86ee19f5ba0670590.jpg": 1, "Leprosy_210823_14_jpg.rf.0a5aabb7460cdce621d3f8c10bb7958a.jpg": 1, "LINE_ALBUM_Leprosy_----------_211004_1_jpg.rf.4f142f913d21685995e82e7b19796306.jpg": 1, "Leprosy_140664_210823_8_jpg.rf.ced13abff76d52a55123c36f2a938d96.jpg": 1, "Leprosy_210823_194_jpg.rf.3845a1b4a8d83482f2cf3bb51cce9c7c.jpg": 1, "Leprosy_210823_123_jpg.rf.55e1abddd83e3afece8fcbd283de115e.jpg": 1, "Leprosy_210823_198_jpg.rf.4cad96cf61987bdf732bf79e5070491b.jpg": 1, "Leprosy150963_210823_62_jpg.rf.92876d9830d2650c685998c30ba22fef.jpg": 1, "90_jpg.rf.210df3aefa327486b3bb3e6930d36ae7.jpg": 1, "Leprosy-40_jpg.rf.834649c6d2165abf9f480fa8fddaaa38.jpg": 1, "Leprosy150963_210823_88_jpg.rf.2c4158b947e81eb896499bd41671f981.jpg": 1, "Leprosy150963_210823_130_jpg.rf.979fccfd0fcf1832b4bd2223a3dae223.jpg": 1, "Leprosy150963_210823_55_jpg.rf.686dbee730071e844559f5ee3e63264a.jpg": 1, "Leprosy_210823_136_jpg.rf.188ade941217aca710f3be5404325e27.jpg": 1, "Leprosy_210823_150_jpg.rf.bac3dbeff3bcfa010b5502b79d4562c0.jpg": 1, "Leprosy-87_jpg.rf.90f4f4d54be3a6ca8966acb3b7de061a.jpg": 1, "Leprosy_----------_210823_21_jpg.rf.7c4d26f7303144f75bf7051e9750d0e1.jpg": 1, "Leprosy-02_jpg.rf.989d7707ea97ad28662cb7854c8d3c6d.jpg": 1, "Leprosy_210823_56_jpg.rf.42c52d97d91f382b934164724715e430.jpg": 1, "Leprosy-42_jpg.rf.f60620e8fb97c6d4629b7fb0e7a06723.jpg": 1, "Leprosy_210823_115_jpg.rf.da971f255230f3c16bcc2839c3e8769e.jpg": 1, "Leprosy_210823_20_jpg.rf.c91cfe13a59e80d97670e8d4495910a7.jpg": 1, "Leprosy_140664_210823_4_jpg.rf.995ef1b109db2739ab033cc6382ae37b.jpg": 1, "Leprosy_210823_162_jpg.rf.fdb118a2d3e1746dd114c2b23721dfb2.jpg": 1, "Leprosy150963_210823_44_jpg.rf.ea65474750500abf3e0bd23931d4b4ff.jpg": 1, "Leprosy_140664_210823_5_jpg.rf.634aa6289e052c8693708566f3890f2b.jpg": 1, "Leprosy_210823_85_jpg.rf.29216a7b56bb5f962fc7cab264a13285.jpg": 1, "Leprosy150963_210823_39_jpg.rf.a94189cb077b6bc58ea8d34c13f30b17.jpg": 1, "Leprosy150963_210823_70_jpg.rf.e3747bea1a94d9f7abc737b785c5cf7b.jpg": 1, "Leprosy_210823_106_jpg.rf.2cf9b4d18181e9db96f1ae9f4e1a2832.jpg": 1, "Leprosy_210823_211_jpg.rf.74501bf935d980e56db18dc8ff614f32.jpg": 1, "Leprosy-61_jpg.rf.ccb9e42c0669225250998f8fcf568cfe.jpg": 1, "Leprosy150963_210823_16_jpg.rf.8351acd80bade984ce99d30ac2a2e726.jpg": 1, "Leprosy_-----------_210823_4_jpg.rf.234de1e1fed5c59e6979a81a2e11b542.jpg": 1, "Leprosy_210823_125_jpg.rf.b30b9faee717c295870ea3cf7deb7abd.jpg": 1, "Leprosy150963_210823_53_jpg.rf.16192462cfd479fe958c33ec105d608b.jpg": 1, "LINE_ALBUM_Leprosy_----------_211004_2_jpg.rf.3e8a468f2408626a463b52fb2bbd01f7.jpg": 1, "Leprosy150963_210823_4_jpg.rf.30cd21f7678b20495f8bf63b21db297a.jpg": 1, "Leprosy_210823_17_jpg.rf.d18e3273c0f58f34cd8e737cecd8a89e.jpg": 1, "Leprosy_210823_105_jpg.rf.96d7eca42a175d3f89f13b3915c907ef.jpg": 1, "Leprosy_210823_72_jpg.rf.d140fe9ff2cf914764e92bd99df9a026.jpg": 1, "Leprosy150963_210823_129_jpg.rf.f382d54c4c8d89795854eea0e712cf94.jpg": 1, "Leprosy_-------_210823_1_jpg.rf.2df3bfe2cf70ea8955950c2f87e2064f.jpg": 1, "Leprosy_210823_96_jpg.rf.be1c581525b5986ad04bb4f7fda61e65.jpg": 1, "Leprosy-66_jpg.rf.c693a38f8c9f38a529d30fa0165b29cf.jpg": 1, "Leprosy_-------_210823_11_jpg.rf.e22e0854fd620cbff9e0048be2ef6db0.jpg": 1, "Leprosy_210823_86_jpg.rf.82b76b18a328d377f61320491b842269.jpg": 1, "Leprosy-23_jpg.rf.12a551b59b6a836b866573f0d3c3a18d.jpg": 1, "Non-lep----------_210823_39_jpg.rf.51806144751d042e57ef4cc7066c3fb8.jpg": 0, "Non-lep----------_210823_55_jpg.rf.84ad7ed1ead609dcfe6fcbba212d5a9f.jpg": 0, "Non_Leprosy_210823_46_jpg.rf.76c20cb340114a98618ade07c3e6b413.jpg": 0, "Non_Leprosy_210823_86_jpg.rf.5bb80a7704ecc6c8615574cad5d074c5.jpg": 0, "Non_Leprosy_210823_100_jpg.rf.2667988249c6af5c8e9831cfc5d85e86.jpg": 0, "Non_Leprosy_210823_18_jpg.rf.34db640c2d8a260e1cb3996c06fd3550.jpg": 0, "Non_Leprosy_210823_62_jpg.rf.c4c4a2e18e6c377d7493343ab0dcbf7b.jpg": 0, "Non_Leprosy_210823_67_jpg.rf.e338ee24c8490ae6bb93d95015779f3e.jpg": 0, "Non_Leprosy_210823_8_jpg.rf.e2d44b96e1bb9b5111b780adec5ba94a.jpg": 0, "Non-lep----------_210823_38_jpg.rf.9fb7e0473f37bd1c9a29d176ca8a7edd.jpg": 0, "Non_Leprosy_210823_43_jpg.rf.23c5148a0a2e578b10914487751c0aaf.jpg": 0, "Leprosy-96_jpg.rf.d819ae1671cf70ac03a2c773a24abdc5.jpg": 1, "Leprosy_-------_210823_12_jpg.rf.cf9de30bf5e422b6a846a21ec34bbdb3.jpg": 1, "Leprosy_210823_204_jpg.rf.f68455d392bf3e83686570b6a423967f.jpg": 1, "Leprosy-15_jpg.rf.3aab768c27f3a53184d33ed0bb156536.jpg": 1, "Leprosy_210823_29_jpg.rf.ee98c4a34fd224506e2055eb99e29367.jpg": 1, "Leprosy150963_210823_90_jpg.rf.b1a8893c8625d255fdcc9429f9a95405.jpg": 1, "Leprosy-83_jpg.rf.06fb806dd0b780a5218292dc17f11d6a.jpg": 1, "Leprosy150963_210823_128_jpg.rf.952fa900e4ec68db00412b638748e4e9.jpg": 1, "Leprosy_210823_196_jpg.rf.2fc1a87dcf965c4be7c9620b7b935771.jpg": 1, "Leprosy_210823_146_jpg.rf.d9b3ef21836ef293f17863ca085e9d6d.jpg": 1, "Leprosy_210823_21_jpg.rf.007926c3ceda31dd10d0febc6eebdcf8.jpg": 1, "Leprosy_210823_11_jpg.rf.b3acd76bd30ffbed0f9bdbe5deb91dce.jpg": 1, "Leprosy-67_jpg.rf.de57dae39fc5c67a5cbfbe07bfb8c086.jpg": 1, "Leprosy_210823_9_jpg.rf.b3c233f10375c2b14ba93cd5c7039d70.jpg": 1, "Leprosy150963_210823_94_jpg.rf.6b43c4631d2898bb3e83ceaa935d5721.jpg": 1, "Leprosy-03_jpg.rf.59f48a4462daea1bae789463b2927d3c.jpg": 1, "Leprosy150963_210823_26_jpg.rf.6b382cb95aa695419de94dfeec629989.jpg": 1, "Leprosy_210823_141_jpg.rf.3853e408185743b8945fee17c1145b95.jpg": 1, "Leprosy-27_jpg.rf.a78a6f1efaef4027b7762a3a83fcdd09.jpg": 1, "Leprosy-04_jpg.rf.9ac511d16713f07ccb72e5b169ff253a.jpg": 1, "Leprosy_-------_210823_4_jpg.rf.307d8e9b1bc231f32b195466be93a53e.jpg": 1, "Leprosy150963_210823_103_jpg.rf.6ab9f9b1229602a11a0d19f8f584cbd3.jpg": 1, "Leprosy150963_210823_38_jpg.rf.4816ad05f7da0a575921ef924b8bc13b.jpg": 1, "Leprosy150963_210823_57_jpg.rf.44db47b82f8a689c1f614135f37d7ba3.jpg": 1, "Leprosy_210823_133_jpg.rf.8100b4a64793b79eb53ac835d678afb3.jpg": 1, "Leprosy_240564_210823_0_jpg.rf.fdc51a5f80493fe50cc71f03220da0e8.jpg": 1, "Leprosy-50_jpg.rf.ca8a3a03409b6e1ba14af0fdc3bd109c.jpg": 1, "Leprosy150963_210823_84_jpg.rf.826b713cfc71bfa779173016864b7fa9.jpg": 1, "Leprosy_210823_127_jpg.rf.e4d8a8eb374befd64e199c755b19d2bd.jpg": 1, "Leprosy_210823_156_jpg.rf.76c7484f7d2f49daa104395ddadab34c.jpg": 1, "Leprosy150963_210823_102_jpg.rf.a525e7dd9393de2b8e1124d75e1d033c.jpg": 1, "Leprosy_210823_62_jpg.rf.ac5062074637128d7984076c2786bb52.jpg": 1, "Leprosy_210823_81_jpg.rf.ef2d8cd123303e74d2af715392ccedf8.jpg": 1, "Leprosy_210823_137_jpg.rf.c5724eeac9cc01e6a714d5765f443831.jpg": 1, "Leprosy_210823_77_jpg.rf.cb91f87f29059d8845a98c2a69db16b5.jpg": 1, "Leprosy_----------_210823_23_jpg.rf.87a2f72a9149ddae77b29c92dd8e634e.jpg": 1, "Leprosy_210823_207_jpg.rf.a01e9b9cf68528f41538ab172094d643.jpg": 1, "Leprosy_210823_54_jpg.rf.ee345e7fbd87658547c1999ea7627c88.jpg": 1, "Leprosy_210823_4_jpg.rf.974c749da2529d2fbe0ac23fc38289a7.jpg": 1, "Leprosy_210823_180_jpg.rf.72651e6dfa14c37baf8eaa815a67807b.jpg": 1, "Leprosy-21_jpg.rf.b2efbc0163d719d6d20abb8c04198957.jpg": 1, "Non-lep----------_210823_19_jpg.rf.fb73b0344bed1b051f2fa26ca1bbf80e.jpg": 0, "Non-lep----------_210823_33_jpg.rf.130a6a316af6be40b61088e5c0711c39.jpg": 0, "Non_Leprosy_210823_68_jpg.rf.d9c15e4749d98ae202cf5a3e74abbcca.jpg": 0, "Non-lep----------_210823_66_jpg.rf.e66b18bbcfb91f1d00ec6c9ba741d9f0.jpg": 0, "Non-lep----------_210823_18_jpg.rf.7488aba99e13fe221a88e53204c1f909.jpg": 0, "Non-lep----------_210823_67_jpg.rf.a617d0233bfffb1c158dfc84076188e4.jpg": 0, "Non_Leprosy_210823_28_jpg.rf.ff0f1c34701a5cdbb37c805d77912752.jpg": 0, "Non_Leprosy_210823_88_jpg.rf.556aa1ead373211f81a217c80bd26145.jpg": 0, "Non_Leprosy_210823_93_jpg.rf.c34d82225ede004a50607ad24927d043.jpg": 0, "Non_Leprosy_210823_70_jpg.rf.a461d139e1636f7278cb388c1e5d6f0e.jpg": 0, "Non-lep----------_210823_29_jpg.rf.a77ecc154eab574637ae4359414ca8e8.jpg": 0, "Non_Leprosy_210823_55_jpg.rf.b7050babf90dadf446f5a4405b02944b.jpg": 0, "Non_Leprosy_210823_12_jpg.rf.e60cd1363bfe1375171043d2d337868d.jpg": 0, "Non-lep----------_210823_64_jpg.rf.23971bd700a56040ba7b26cd59a0d24a.jpg": 0, "Non_Leprosy_210823_37_jpg.rf.ea12e0968976cb047db137773dd4f5e8.jpg": 0, "Non-lep----------_210823_24_jpg.rf.6dd1351f702e7632afbfb93ea5461224.jpg": 0, "Non_Leprosy_210823_40_jpg.rf.a2e14072fbc0112c594219d71bd540ae.jpg": 0, "Non_Leprosy_210823_15_jpg.rf.b0f7cba18533a0de03f93faf939a6ded.jpg": 0, "Non-lep----------_210823_73_jpg.rf.de30b51645b399cccb0959c4fb882fa2.jpg": 0}')
found = {}
for s in ["test", "valid"]:
    for root, _, files in os.walk(os.path.join(EXTERNAL_DIR, s)):
        for f in files:
            if f in EXT: found[f] = os.path.join(root, f)
missing = [f for f in EXT if f not in found]; print("صور خارجية مفقودة:", len(missing))
names = sorted(found); p_ext = predict_paths([found[n] for n in names]); y_ext = np.array([EXT[n] for n in names])
pd.DataFrame({"filename": names, "y": y_ext, "p_leprosy": p_ext}).to_csv(os.path.join(OUT_DIR, "external_probs.csv"), index=False)

def report(name, y, p, th=0.5):
    pr = (p >= th).astype(int); tp=((pr==1)&(y==1)).sum(); fn=((pr==0)&(y==1)).sum(); fp=((pr==1)&(y==0)).sum(); tn=((pr==0)&(y==0)).sum()
    print(f"{name}: n={len(y)} acc={(tp+tn)/len(y)*100:.2f} recall={tp/(tp+fn)*100:.2f} spec={tn/(tn+fp)*100:.2f} AUC={roc_auc_score(y,p):.3f}  TP/FN/FP/TN={tp}/{fn}/{fp}/{tn}")
report("INTERNAL test", y, p_int); report("EXTERNAL Roboflow", y_ext, p_ext)

import zipfile; from google.colab import files
z = os.path.join("/content", f"stage1_{VARIANT}_results.zip")
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["test_probs.csv", "external_probs.csv", "history.csv"]: zf.write(os.path.join(OUT_DIR, f), f)
files.download(z)